# Within-region modelling (Sec. 3.1; Fig. 3, A2, A3a, A4)

Trains and evaluates one LSTM and one Transformer per region, each trained and tested within the same region on held-out glaciers.

| Paper figure | Content | Section | Saved as (in `figures/paperTF/`) |
|---|---|---|---|
| Fig. 3 | Transformer, modeled vs observed PMB per region | Fig. 3 | `fig3_within_region_transformer_only.png` |
| Fig. A2 | LSTM vs Transformer, modeled vs observed PMB per region | Fig. A2 | `appendix/A2_within_region_lstm_vs_transf_mixed.png` |
| Fig. A3a | Transformer for Central Asia sub-region 3, coloured by glacier | Fig. A3a | `appendix/A3_a.png` + `appendix/A3_a_legend.png` |
| Fig. A4 | Sinkhorn-informed vs random glacier-level splits | Fig. A4 | `appendix/A4_random_vs_smart_split_transformer.png` |

**Steps**
1. Load the stake datasets and their monthly ERA5-Land inputs (`build_monthly_cache`, from `1.4 Create_grids`).
2. Split Central Asia, Alaska and Italy/Austria into sub-regions.
3. Make the train/test splits: a Sinkhorn-informed glacier-level (GL) split per region, a measurement-level (ID) split for regions with few glaciers, and 5 random GL splits per region as a baseline.
4. Build the LSTM/Transformer datasets, train (or load) the models and evaluate them on the test sets.

Splits, datasets and models are cached under `cfg.dataPath/cache/TF_Europe/WITHIN_REGION/` and only recomputed when missing (or when forced, see the flags in each section). The functions used here live in `scripts/`, mainly `models/within_region.py` (figures, split comparison) and `dataset/within_region_splits.py` (split checkpoints).

## Setting up:

In [ ]:
import os

# Silence KeOps (used by geomloss for Sinkhorn distances) before it is imported
os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

import torch

# Initialise CUDA before the other imports: changing this order changes the
# GPU results slightly (~1e-7), so keep it to reproduce the paper figures exactly
if torch.cuda.is_available():
    torch.cuda.init()

import sys
import copy
import pickle
import logging
import warnings
from pathlib import Path

import pandas as pd
import massbalancemachine as mbm

sys.path.append(os.path.join(os.getcwd(), '../../'))  # Add root of repo to import MBM
# Shared config (paths, region lists, IT_GLACIERS/AT_GLACIERS, ...) and model settings
# (PARAMS_LSTM, PARAMS_TRANSFORMER, N_EPOCHS, MODEL_DATE)
from papers.Transformer_TF.scripts.config_TF_Europe import *
from papers.Transformer_TF.scripts.config_models import *
from papers.Transformer_TF.scripts.dataset import *
from papers.Transformer_TF.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

# Call config file (data paths, seed, etc.)
cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)  # reproducibility
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

# Use the GPU if one is visible
device = torch.device("cuda" if torch.cuda.device_count() > 0 else "cpu")
print(f"Device: {device}")

In [ ]:
# Cache folder for the splits, datasets and models of this notebook
BASE_DIR = Path(cfg.dataPath) / path_cache / "WITHIN_REGION"
print(f"Base directory for data: {BASE_DIR}")
os.makedirs(BASE_DIR, exist_ok=True)

## Data:
### Stakes and monthly inputs:
Loads the stake datasets of all regions in `RGI_REGIONS` and their monthly inputs (ERA5-Land climate + topography per stake and month). With `run_flag_by_code = False` the monthly data are loaded from `WGMS/CrossRegional/monthly_cache/` instead of being recomputed.

In [ ]:
# Stake datasets of all regions configured in RGI_REGIONS (one DataFrame per RGI region)
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

# Region codes of the monthly cache (before splitting into sub-regions)
WITHIN_CODES = ["CH", "NOR", "ISL", "FR", "IT_AT", "SJM", "CENTRALASIA", "ALA"]

run_flag_by_code = {code: False for code in WITHIN_CODES}  # True = recompute

monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=VOIS_CLIMATE,
    vois_topographical=STATIC_COLS,
    region_codes=WITHIN_CODES,
    run_flag_by_code=run_flag_by_code,
)

### Sub-regions:
Central Asia and Alaska are split by RGI O2 sub-region (CA_12, CA_3, CA_4; ALA_2, ALA_4, ALA_6), and Italy/Austria by country (IT, AT). The parent entries CENTRALASIA and ALA are kept; IT_AT is dropped.

In [ ]:
# Central Asia: add the O2 sub-region of each glacier, then split
dfs, glacier_dict_ca = add_o2region_to_dfs(dfs, '13', cfg)
monthly_cache = add_o2region_to_monthly_cache(monthly_cache, 'CENTRALASIA',
                                              glacier_dict_ca)
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'CENTRALASIA',
    subregions={
        'CA_12': {
            'o2region_col': 'O2Region',
            'values': ['1', '2']
        },
        'CA_3': {
            'o2region_col': 'O2Region',
            'values': ['3']
        },
        'CA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
    },
)

# Alaska: same, deduplicating glaciers listed under several names
dfs, glacier_dict_alaska = add_o2region_to_dfs(dfs,
                                               '01',
                                               cfg,
                                               deduplicate_glaciers=True)
monthly_cache = add_o2region_to_monthly_cache(monthly_cache, 'ALA',
                                              glacier_dict_alaska)
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'ALA',
    subregions={
        'ALA_2': {
            'o2region_col': 'O2Region',
            'values': ['2']
        },
        'ALA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
        'ALA_6': {
            'o2region_col': 'O2Region',
            'values': ['6']
        },
    },
)

# Italy / Austria: split by country (glacier lists from the config)
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'IT_AT',
    subregions={
        'IT': IT_GLACIERS,
        'AT': AT_GLACIERS
    },
    drop_parent=True,
)

# Number of glaciers per (sub-)region
for region in monthly_cache.keys():
    print("{}: {} glaciers".format(
        region, len(monthly_cache[region]['data_monthly'].GLACIER.unique())))

## Train/test splits:
### Sinkhorn-informed splits:
For each region, the glacier-level (GL) split minimising the Sinkhorn distance between the train and test feature distributions is selected from 100 random candidates (80/20 in number of measurements; Sec. 2.4). Regions with too few glaciers (Svalbard, Central Asia SR 3–4, Alaska) also get a measurement-level (ID) split.

The splits are cached per region in checkpoints and together in `within_region_splits_cache.pkl`. To recompute some regions, list them in `RECOMPUTE_CODES`.

In [ ]:
RECOMPUTE_CODES = []  # region codes to recompute; empty list = load everything from cache

for p in [
        BASE_DIR / "within_region_splits_checkpoint_gl.pkl",
        BASE_DIR / "within_region_splits_checkpoint_id.pkl",
]:
    purge_codes_from_checkpoint(p, RECOMPUTE_CODES)

# Ignore the combined cache if anything is recomputed, so it reflects the fresh run
FORCE_RERUN_OUTER = bool(RECOMPUTE_CODES)

# Region codes after the sub-region split
WITHIN_CODES_ALL = [
    c for c in WITHIN_CODES if c not in {'IT_AT', 'CENTRALASIA'}
] + ['IT', 'AT', 'CA_12', 'CA_3', 'CA_4']

SPLITS_CACHE = BASE_DIR / "within_region_splits_cache.pkl"

# Regions split by glacier (GL) where possible, and regions that also get an ID split
GL_LEVEL_SPLIT = [
    'CH', 'NOR', 'ISL', 'FR', 'IT', 'AT', 'ALA', 'CA_3', 'CA_4', 'SJM'
]
ID_LEVEL_SPLIT = ['SJM', 'CA_3', 'CA_4', 'ALA']

if FORCE_RERUN_OUTER or not SPLITS_CACHE.exists():
    res_within_by_code_gl, split_info_by_code_gl = prepare_within_region_pairs_from_cache(
        cfg=cfg,
        monthly_cache=monthly_cache,
        within_codes=WITHIN_CODES_ALL,
        gl_level_split=GL_LEVEL_SPLIT,
        checkpoint_path=BASE_DIR / "within_region_splits_checkpoint_gl.pkl",
        seed=0)

    res_within_by_code_id, split_info_by_code_id = prepare_within_region_pairs_from_cache(
        cfg=cfg,
        monthly_cache=monthly_cache,
        within_codes=ID_LEVEL_SPLIT,
        gl_level_split=[],
        checkpoint_path=BASE_DIR / "within_region_splits_checkpoint_id.pkl",
        seed=0)

    with open(SPLITS_CACHE, "wb") as f:
        pickle.dump(
            {
                "res_within_by_code_gl": res_within_by_code_gl,
                "split_info_by_code_gl": split_info_by_code_gl,
                "res_within_by_code_id": res_within_by_code_id,
                "split_info_by_code_id": split_info_by_code_id,
            }, f)
    print(f"Saved splits to cache: {SPLITS_CACHE}")
else:
    with open(SPLITS_CACHE, "rb") as f:
        cache = pickle.load(f)
    res_within_by_code_gl = cache["res_within_by_code_gl"]
    split_info_by_code_gl = cache["split_info_by_code_gl"]
    res_within_by_code_id = cache["res_within_by_code_id"]
    split_info_by_code_id = cache["split_info_by_code_id"]
    print(f"Loaded splits from cache: {SPLITS_CACHE}")

### Random splits:
Baseline for Fig. A4: 5 random glacier-level splits per region, with the same regions and 80/20 target as above.

In [ ]:
RECOMPUTE_RANDOM_SPLITS = False  # True = delete the checkpoint and recompute
if RECOMPUTE_RANDOM_SPLITS:
    p = BASE_DIR / "within_region_random_splits_checkpoint.pkl"
    if p.exists():
        os.remove(p)
        print(f"Deleted {p}")

RANDOM_SPLITS_CACHE = BASE_DIR / "within_region_random_splits_cache.pkl"
N_RANDOM_SPLITS = 5

if RECOMPUTE_RANDOM_SPLITS or not RANDOM_SPLITS_CACHE.exists():
    res_within_by_code_random, split_info_by_code_random = prepare_within_region_random_splits(
        cfg=cfg,
        monthly_cache=monthly_cache,
        within_codes=WITHIN_CODES_ALL,
        gl_level_split=GL_LEVEL_SPLIT,
        n_random_splits=N_RANDOM_SPLITS,
        checkpoint_path=BASE_DIR /
        "within_region_random_splits_checkpoint.pkl",
    )

    with open(RANDOM_SPLITS_CACHE, "wb") as f:
        pickle.dump(
            {
                "res_within_by_code_random": res_within_by_code_random,
                "split_info_by_code_random": split_info_by_code_random,
            }, f)
    print(f"Saved random splits to cache: {RANDOM_SPLITS_CACHE}")
else:
    with open(RANDOM_SPLITS_CACHE, "rb") as f:
        cache_random = pickle.load(f)
    res_within_by_code_random = cache_random["res_within_by_code_random"]
    split_info_by_code_random = cache_random["split_info_by_code_random"]
    print(f"Loaded random splits from cache: {RANDOM_SPLITS_CACHE}")

## Models:
### Model datasets:
All splits are merged under one set of keys: the plain region code (or `<code>_GL` / `<code>_ID` for regions with both split types) and `<code>_rand<i>` for the random splits. A dataset is then built (or loaded from `datasets/`) for each key.

In [ ]:
res_within_by_code, split_info_by_code = combine_within_region_splits(
    res_within_by_code_gl,
    split_info_by_code_gl,
    res_within_by_code_id,
    split_info_by_code_id,
    res_within_by_code_random,
    split_info_by_code_random,
)

logs_cache_dir = BASE_DIR / "datasets"
logs_cache_dir.mkdir(parents=True, exist_ok=True)

lstm_assets = build_or_load_lstm_all(
    cfg=cfg,
    res_all=res_within_by_code,
    MONTHLY_COLS=MONTHLY_COLS,
    STATIC_COLS=STATIC_COLS,
    cache_dir=logs_cache_dir,
    force_recompute=False,
    include_atomic=True,
    include_groups=True,
)

### Parameters:
The same hyperparameters (`PARAMS_LSTM`, `PARAMS_TRANSFORMER` in `scripts/config_models.py`) are used for every region; they can be changed per key here.

In [ ]:
lstm_params_by_key = {key: copy.deepcopy(PARAMS_LSTM) for key in lstm_assets.keys()}
transformer_params_by_key = {
    key: copy.deepcopy(PARAMS_TRANSFORMER)
    for key in lstm_assets.keys()
}

### Train models:
One LSTM and one Transformer per key. Trained models are saved in `models/` and loaded on later runs (`force_retrain=False`).

In [ ]:
# LSTM
models_lstm, infos_lstm = train_within_region_models_all(
    cfg=cfg,
    lstm_assets_by_key=lstm_assets,
    params_by_key=lstm_params_by_key,
    device=device,
    epochs=N_EPOCHS,
    force_retrain=False,
    models_dir=BASE_DIR / "models",
    prefix="lstm_within",
    date=MODEL_DATE,
    model_type="lstm",
)

In [ ]:
# Transformer
models_tf, infos_tf = train_within_region_models_all(
    cfg=cfg,
    lstm_assets_by_key=lstm_assets,
    params_by_key=transformer_params_by_key,
    device=device,
    epochs=N_EPOCHS,
    force_retrain=False,
    models_dir=BASE_DIR / "models",
    prefix="transformer_within",
    date=MODEL_DATE,
    model_type="transformer",
)

In [ ]:
# Number of trainable parameters (same for all regions, as the inputs are the same)
df_params = pd.DataFrame({
    "LSTM": {k: count_trainable_params(m) for k, m in models_lstm.items()},
    "Transformer": {k: count_trainable_params(m) for k, m in models_tf.items()},
})
print(df_params.describe().loc[["min", "max"]])
df_params.head()

### Evaluate on test:
Metrics (RMSE, R², ...) of every model on its test set, for the Transformer (`df_metrics`) and the LSTM (`df_metrics_lstm`). These are also used for the split comparison (Fig. A4).

In [ ]:
df_metrics, preds_by_key, figs_by_key, fig_grid = evaluate_all_models_within_reg(
    cfg=cfg,
    models_by_key=models_tf,
    lstm_assets_by_key=lstm_assets,
    device=device,
    ax_xlim=(-16, 10),
    ax_ylim=(-16, 10),
    grid_shape=(2, 3),
    grid_figsize=(25, 12),
    complement_key="",
)

df_metrics_lstm, preds_by_key_lstm, figs_by_key_lstm, fig_grid_lstm = evaluate_all_models_within_reg(
    cfg=cfg,
    models_by_key=models_lstm,
    lstm_assets_by_key=lstm_assets,
    device=device,
    ax_xlim=(-16, 10),
    ax_ylim=(-16, 10),
    grid_shape=(2, 3),
    grid_figsize=(25, 12),
    complement_key="",
)

## Paper figures:
Regions shown in Fig. 3 and A2, and the model key used for each (GL split; Svalbard and Central Asia SR 4 are not shown, as their few glaciers do not allow a glacier-level split).

In [ ]:
region_to_key = {
    'CH': 'CH',
    'FR': 'FR',
    'IT': 'IT',
    'AT': 'AT',
    'NOR': 'NOR',
    'ISL': 'ISL',
    'ALA': 'ALA_GL',
    'CA_3': 'CA_3_GL',
}

### Fig. A2: LSTM vs Transformer

In [ ]:
plot_within_region_grid(
    cfg,
    device,
    region_to_key,
    models_lstm=models_lstm,
    models_tf=models_tf,
    lstm_assets=lstm_assets,
    out_path="figures/paperTF/appendix/A2_within_region_lstm_vs_transf_mixed.png",
)

### Fig. 3: Transformer only

In [ ]:
plot_within_region_grid_transformer_only(
    cfg,
    device,
    region_to_key,
    models_tf=models_tf,
    lstm_assets=lstm_assets,
    out_path="figures/paperTF/fig3_within_region_transformer_only.png",
)

### Fig. A3a: Central Asia sub-region 3 by glacier
The map of the same test glaciers (Fig. A3b) is made in `2.1 Study_regions`, with the same colours.

In [ ]:
CA3_KEY = "CA_3_GL"  # glacier-level split of Central Asia sub-region 3

test_df_preds_ca3, fig_ca3, ax_ca3, color_by_glacier, legend_fig = plot_region_pred_vs_truth_by_glacier(
    cfg=cfg,
    model=models_tf[CA3_KEY],
    device=device,
    lstm_assets_for_key=lstm_assets[CA3_KEY],
    ax_xlim=(-8, 4),
    ax_ylim=(-8, 4),
    legend_separate=True)

# save figure and legend separately (combined into Fig. A3 outside the code)
fig_ca3.savefig("figures/paperTF/appendix/A3_a.png",
                dpi=NATURE_SPECS["dpi"],
                bbox_inches="tight")
legend_fig.savefig("figures/paperTF/appendix/A3_a_legend.png",
                   dpi=NATURE_SPECS["dpi"],
                   bbox_inches="tight")
plt.show()

### Fig. A4: Sinkhorn-informed vs random splits
For each region and metric, the Sinkhorn split is compared with the mean over the 5 random splits (`delta` > 0: Sinkhorn split better). Svalbard and Central Asia SR 4 are excluded (no glacier-level split).

In [ ]:
df_split_cmp = pd.concat(
    [
        build_split_comparison(df_metrics, "Transformer"),
        build_split_comparison(df_metrics_lstm, "LSTM"),
    ],
    ignore_index=True,
)
display(df_split_cmp)

In [ ]:
plot_split_comparison(
    df_split_cmp,
    out_path="figures/paperTF/appendix/A4_random_vs_smart_split_transformer.png",
    model="Transformer",
    metrics=("RMSE_annual", "RMSE_winter"),
    region_order=("CH", "FR", "IT", "AT", "NOR", "ISL", "ALA", "CA_3"),
)